# 06 · Modeling

**Phase skill: fitting a regression — and noticing when it is lying to you.**

The question: can a monster's printed LV be predicted from its statblock? You will fit
the model twice. Trust the process; do **not** skip ahead to the reveal cell.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression

sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))
from analysis import load_sd_features

DB_PATH = (Path.cwd().parent / "data" / "monsterlab.db").resolve()
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
df = load_sd_features(conn)
print(f"{len(df)} monsters loaded")

## Exercise 6.1 — fit with everything

Fit a `LinearRegression` predicting `level` from all six candidate features:
`ac`, `hp`, `best_attack_bonus`, `best_avg_damage`, `best_num_attacks`,
`best_stat_mod`.

Data note: a missing `best_avg_damage` or `best_attack_bonus` means the monster has no
damage dice or no listed bonus — those are real zeros, not missing data, so **fill NaN
feature values with 0** rather than dropping rows.

**Produce:** `model_full` — the fitted model — and `r2_full` — its R² on the data it
was fit on.

<details><summary>Hint 1 (concept)</summary>

R-squared is the fraction of the target's variance the features explain; sklearn models can report it about their own training data.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `DataFrame.fillna`, `LinearRegression.fit`, and `.score`.

</details>

In [ ]:
FEATURES_ALL = [
    "ac", "hp", "best_attack_bonus", "best_avg_damage", "best_num_attacks", "best_stat_mod",
]

# Produce: model_full (fitted LinearRegression), r2_full (float)

model_full = ...
r2_full = ...

print(f"R-squared = {r2_full:.4f}")

In [ ]:
assert 0 <= r2_full <= 1, f"r2_full = {r2_full!r} -- that is not an R-squared"
assert abs(r2_full - 0.997) < 0.003, (
    f"r2_full = {r2_full:.4f}, expected about 0.997 -- all six features in, NaNs filled with 0, target is level")
print(f"PASS: R-squared = {r2_full:.3f}.")
print("Context: six statblock numbers explain 99.7% of printed LV. On 243 messy hand-written monsters. Hold that thought.")

## Stop. Why should this worry you?

An R² of 0.997 on real, hand-authored game data. Before reading anything further, write
your honest answer: is this model *good*, or is something else going on? What would you
check first?

*Your answer (write it before opening the next cell):*

*…*

## The reveal

<details><summary>Open after you have written your answer above.</summary>

Shadowdark monsters get **one hit die per level** — HP is generated *from* LV when a
monster is written. So `hp` isn't evidence about the target; it is the target, wearing
a paper-thin disguise (`hp ≈ 4.5 × level`, and 5.4 shows corr ≈ 0.998). The regression
isn't predicting LV from stats — it is reading LV off the sheet and dividing by 4.5.

This is **target leakage**: a feature that is causally *downstream* of the label, so it
smuggles the answer into the inputs. Leaky models post spectacular training numbers and
then collapse the moment they meet data where the leak doesn't hold — here, e.g., a
homebrew monster whose author didn't follow the hit-die convention.

The fix is not statistical, it's causal: ask of every feature, *"could I know this
before knowing the answer?"* For `hp` in Shadowdark, the answer is no. Drop it.

</details>

## Exercise 6.2 — refit without the leak

Refit the same regression with `hp` removed from the feature list (same fill-with-0
handling).

**Produce:** `FEATURES_NO_HP` (the five-feature list), `model_nohp` (fitted model), and
`r2_nohp` (its R² on the fitting data).

<details><summary>Hint 1 (concept)</summary>

Expect the number to drop. That drop is the honest gap between what the statblock tells you and what was leaking.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Same tools as 6.1 -- only the feature list changes.

</details>

In [ ]:
# Produce: FEATURES_NO_HP (list), model_nohp (fitted LinearRegression), r2_nohp (float)

FEATURES_NO_HP = ...
model_nohp = ...
r2_nohp = ...

print(f"R-squared without hp = {r2_nohp:.4f}")

In [ ]:
assert "hp" not in FEATURES_NO_HP and len(FEATURES_NO_HP) == 5, (
    f"FEATURES_NO_HP = {FEATURES_NO_HP!r} -- it should be the six features minus hp")
assert abs(r2_nohp - 0.8175) < 0.005, (
    f"r2_nohp = {r2_nohp:.4f}, expected about 0.82 -- same setup as 6.1, five features")
print(f"PASS: R-squared = {r2_nohp:.2f} without the leak.")
print("Context: 0.82 is the real signal -- a statblock mostly determines its LV, with honest room for design judgment.")

## Exercise 6.3 — coefficients and residuals

Two reads on the honest model:

1. **Coefficients.** Extract `model_nohp`'s coefficients into a Series indexed by
   feature name.
2. **Residuals.** Define `residual = level − predicted_level` for every monster. The
   five most **negative** residuals *punch above their weight* (their stats justify a
   higher LV than printed); the five most **positive** *punch below it*.

**Produce:**

- `coefs` — Series of the five coefficients, indexed by feature name,
- `punch_above` — `set` of the 5 monster names with the most negative residuals,
- `punch_below` — `set` of the 5 names with the most positive residuals.

<details><summary>Hint 1 (concept)</summary>

A linear coefficient answers: holding the other features fixed, one unit more of this feature moves the prediction by how many LV?

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `model.coef_` paired with the feature list, `model.predict`, and pandas `nsmallest` / `nlargest`.

</details>

In [ ]:
# Produce: coefs (Series), punch_above (set of 5 names), punch_below (set of 5 names)

coefs = ...
punch_above = ...
punch_below = ...

print(coefs)
print("above:", sorted(punch_above))
print("below:", sorted(punch_below))

In [ ]:
assert set(coefs.index) == set(FEATURES_NO_HP), f"coefs is indexed by {list(coefs.index)}, expected the five feature names"
top_feature = coefs.abs().idxmax()
assert top_feature == "best_attack_bonus", (
    f"the largest-magnitude coefficient belongs to {top_feature!r} -- check the pairing of coef_ with feature names")
assert abs(coefs["best_attack_bonus"] - 0.948) < 0.01, (
    f"coef(best_attack_bonus) = {coefs['best_attack_bonus']:.3f}, expected about 0.95")
want_above = {"Giant, Fire", "Gorilla", "Lesser Elemental, Air", "Lesser Elemental, Earth", "Lesser Elemental, Fire"}
want_below = {"Druid", "Mordanticus The Flayed", "Rathgamnon", "The Tarrasque", "Viperian, Wizard"}
assert isinstance(punch_above, set) and isinstance(punch_below, set), "punch_above and punch_below should be sets of names"
assert punch_above == want_above, (
    f"punch_above matches {len(punch_above & want_above)} of the 5 expected names -- "
    "check the residual sign convention and that predictions come from model_nohp")
assert punch_below == want_below, (
    f"punch_below matches {len(punch_below & want_below)} of the 5 expected names -- "
    "check the residual sign convention and that predictions come from model_nohp")
print("PASS: coefficients and both residual top-5s match the pipeline.")
print("Context: every +1 attack bonus is worth about +0.95 LV -- the single loudest signal in a statblock.")

**Interpretation prompts** (a sentence each):

1. In plain GM terms, what does the ≈0.95 coefficient on attack bonus mean?

*…*

2. Three of the five "punch above their weight" monsters are Lesser Elementals. One
   sentence: coincidence, or does that pattern smell like a design decision?

*…*

Notebook 07 asks the question every model must survive: does any of this hold on data
the model never saw?